In [1]:
import pandas as pd
from sklearn.cluster import KMeans

In [3]:
df = pd.read_csv("/content/nba_players_35_years_stats_and_bio.csv")

/tmp/ipykernel_9517/3513661895.py:1: DtypeWarning: Columns (68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/nba_players_35_years_stats_and_bio.csv")


In [4]:
df

,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE_x,GP,W,L,W_PCT,...,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT,PLAYER_HEIGHT_INCHES,PLAYER_WEIGHT,AGE_y,Season
0,920,A.C. Green,A.C.,1610612742,DAL,33.0,83,23,60,0.277,...,394,155,49,24,165,2.0,81.0,225,33.0,1996-97
1,243,Aaron McKie,Aaron,1610612765,DET,24.0,83,48,35,0.578,...,93,229,168,24,233,2.0,77.0,209,24.0,1996-97
2,1425,Aaron Williams,Aaron,1610612763,VAN,25.0,33,4,29,0.121,...,363,207,219,24,213,2.0,81.0,225,25.0,1996-97
3,768,Acie Earl,Acie,1610612749,MIL,27.0,47,14,33,0.298,...,269,304,219,24,307,2.0,83.0,240,27.0,1996-97
4,228,Adam Keefe,Adam,1610612762,UTA,27.0,62,48,14,0.774,...,77,282,168,24,288,1.0,81.0,241,27.0,1996-97
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10340,1629121,Jaylen Adams,Jaylen,1610612737,ATL,23.0,34,13,21,0.382,...,379,406,261,38,403,1.0,74.0,190.0,23.0,2018-19
10341,1627759,Jaylen Brown,Jaylen,1610612738,BOS,22.0,74,41,33,0.554,...,156,154,159,38,139,1.0,79.0,220.0,22.0,2018-19
10342,1628537,Jaylen Morris,Jaylen,1610612749,MIL,23.0,4,4,0,1.000,...,58,448,261,38,458,1.0,77.0,185.0,23.0,2018-19
10343,1628369,Jayson Tatum,Jayson,1610612738,BOS,21.0,79,48,31,0.608,...,34,75,82,38,73,1.0,80.0,208.0,21.0,2018-19


In [5]:
df['Player'] = df['PLAYER_NAME'] + ' ' + df['Season']

In [15]:
df['AGE'] = df['AGE_x']

In [24]:
df_final = df[['Player', 'AGE', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M',\
   'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',\
   'STL', 'BLK', 'PF', 'PTS']]

In [25]:
df_final = df_final.dropna()

In [27]:
df_final = df_final.set_index("Player")

In [28]:
df_final

,AGE,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,PF,PTS
Player,,,,,,,,,,,,,,,,,,,
A.C. Green 1996-97,33.0,30.1,2.8,5.8,0.483,0.0,0.2,0.050,1.5,2.4,0.650,2.7,5.2,7.9,0.8,0.8,0.2,1.7,7.2
Aaron McKie 1996-97,24.0,19.6,1.8,4.4,0.411,0.5,1.2,0.398,1.1,1.3,0.836,0.5,2.2,2.7,1.9,0.9,0.3,1.6,5.2
Aaron Williams 1996-97,25.0,17.0,2.6,4.5,0.574,0.0,0.0,0.000,1.0,1.5,0.673,1.9,2.5,4.3,0.5,0.5,0.9,2.2,6.2
Acie Earl 1996-97,27.0,10.6,1.4,3.8,0.374,0.0,0.1,0.000,1.1,1.8,0.643,0.7,1.3,2.0,0.4,0.3,0.6,1.3,4.0
Adam Keefe 1996-97,27.0,14.8,1.3,2.6,0.513,0.0,0.0,0.000,1.1,1.7,0.689,1.2,2.3,3.5,0.5,0.5,0.2,1.6,3.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Jawun Evans 2018-19,22.0,8.0,0.4,1.8,0.214,0.0,0.4,0.000,0.0,0.0,0.000,0.1,1.4,1.5,1.3,0.4,0.0,1.0,0.8
Jaylen Adams 2018-19,23.0,12.6,1.1,3.2,0.345,0.7,2.2,0.338,0.2,0.3,0.778,0.3,1.4,1.8,1.9,0.4,0.1,1.3,3.2
Jaylen Brown 2018-19,22.0,25.9,5.0,10.7,0.465,1.3,3.7,0.344,1.8,2.7,0.658,0.9,3.4,4.2,1.4,0.9,0.4,2.5,13.0


In [46]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# נניח שכבר הגדרת את 'Player' בתור האינדקס וביצעת נרמול (Scaling)
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_final)

def get_top_5_similar_players(player_name, df, scaled_data):
    # 1. נוודא שהשחקן אכן קיים בנתונים כדי למנוע שגיאות
    if player_name not in df.index:
        return "שחקן לא נמצא במסד הנתונים."

    # 2. מציאת האינדקס (מספר השורה) של השחקן המבוקש
    player_idx = df.index.get_loc(player_name)

    # שליפת וקטור הנתונים המנורמלים של השחקן
    player_vector = scaled_data[player_idx].reshape(1, -1)

    # 3. חישוב דמיון הקוסינוס בין השחקן לכל שאר השחקנים במסד הנתונים
    # מחזיר מערך של ציונים בין 1- ל-1
    similarities = cosine_similarity(player_vector, scaled_data).flatten()

    # 4. מציאת האינדקסים של השחקנים עם הציון הגבוה ביותר
    # argsort מסדר מהנמוך לגבוה, אז ניקח את ה-6 האחרונים ונהפוך את הסדר
    # (לוקחים 6 כי הראשון תמיד יהיה השחקן עצמו עם ציון של 1.0)
    top_indices = similarities.argsort()[-6:][::-1][1:]

    # 5. ארגון התוצאות לתוך טבלה מסודרת
    results = []
    for idx in top_indices:
        match_name = df.index[idx]
        score = similarities[idx]
        results.append({'Player': match_name, 'Similarity Score': round(score, 4)})

    return pd.DataFrame(results)